## Importing Libraries

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Load environment variables from .env file
load_dotenv()
# Initialize the OpenAI API client
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

### 1. ZERO-SHOT

In [4]:
print("=== 1. Zero-Shot ===")
zero_shot = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always respond with valid JSON only."),
    ("human", """Classify the sentiment of this text as POSITIVE, NEGATIVE, or NEUTRAL.
Return ONLY a JSON object like: {{"sentiment": "POSITIVE", "confidence": "high"}}

Text: '{text}'""")
])

chain = zero_shot | llm | JsonOutputParser()

print(chain.invoke({"text": "The deployment went smoothly but the docs were terrible."}))
print()

=== 1. Zero-Shot ===
{'sentiment': 'NEUTRAL', 'confidence': 'low'}



### 2. FEW-SHOT

In [6]:
print("=== 2. Few-Shot ===")
few_shot = ChatPromptTemplate.from_messages([
    ("system", "Classify API errors. Learn the pattern from these examples."),
    ("human", "Error: Connection refused on port 5432"),
    ("ai", "CATEGORY: DB_CONNECTION | SEVERITY: HIGH | ACTION: Check if PostgreSQL is running"),
    ("human", "Error: JWT token expired"),
    ("ai", "CATEGORY: AUTH_EXPIRY | SEVERITY: LOW | ACTION: Refresh the access token"),
    ("human", "Error: Rate limit exceeded, retry after 60s"),
    ("ai", "CATEGORY: RATE_LIMIT | SEVERITY: MEDIUM | ACTION: Implement exponential backoff"),
    ("human", "Error: {error_message}"),
])

chain = few_shot | llm | StrOutputParser()
print(chain.invoke({"error_message": "Redis NOAUTH Authentication required"}))
print()

=== 2. Few-Shot ===
CATEGORY: AUTH_REQUIRED | SEVERITY: HIGH | ACTION: Authenticate with Redis using the AUTH command (provide the correct password) before issuing commands; verify the server's requirepass and client credentials.



### 3. CHAIN-OF-THOUGHT (ZERO-SHOT)

In [7]:
print("=== 3. Chain-of-Thought (Zero-Shot) ===")
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a backend architecture expert."),
    ("human", """
Our SaaS API:
- Currently: single FastAPI server, PostgreSQL, no cache
- Current load: 1,000 requests/day
- Expected: 100x growth in 3 months (100,000 requests/day)
- Budget: minimal (startup)

What changes are needed? Think through this step by step.
""")
])

chain = cot_prompt | llm | StrOutputParser()
print(chain.invoke({}))
print()

=== 3. Chain-of-Thought (Zero-Shot) ===
Step 1 — establish baseline and goals
- Current: 1,000 requests/day (~0.7 req/sec on average). Likely low concurrency, but you’ll have hot paths and DB bottlenecks can emerge at scale.
- Target: 100,000 requests/day (~1.2 req/sec average, with spikes). Plan for higher concurrency than today (trust, not just average).
- Goals (short list): keep latency low (p95/p99 acceptable), keep error rate very low, keep total cost modest, keep deployment/simple operations.

Step 2 — identify the bottlenecks to address first
- API server (FastAPI):
  - Are you using async endpoints? CPU-bound work should be offloaded.
  - How many workers? Is the current setup constrained by CPU, memory, or slow DB calls?
- PostgreSQL:
  - Connection limits and query performance (missing indexes, N+1 queries, slow scans).
  - Does the app overwhelm the DB with too many connections?
- Caching:
  - Absence of cache means repeated misses for common reads can hurt latency.
- Obser

### 4. CHAIN-OF-THOUGHT (FEW-SHOT)

In [8]:
print("=== 4. Chain-of-Thought (Few-Shot) ===")
cot_few = ChatPromptTemplate.from_messages([
    ("system", "You debug slow database queries step by step."),
    ("human", "Query: SELECT * FROM users WHERE email = 'x@x.com'. Takes 3 seconds. 2M rows. No index on email."),
    ("ai", """
Step 1 — Identify bottleneck:
  3 seconds on 2M rows = sequential scan. PostgreSQL reads every row.
  Root cause: no index on the email column used in WHERE.

Step 2 — Choose the fix:
  email is high-cardinality (each value unique) = ideal for B-tree index.
  B-tree reduces lookup from O(n) to O(log n).

Step 3 — Verify side effects:
  Index adds ~100MB storage. INSERT/UPDATE ~5% slower. Acceptable tradeoff.

Fix: CREATE INDEX CONCURRENTLY idx_users_email ON users(email);
CONCURRENTLY avoids table lock — safe for production.
Expected result: query drops from 3s to <10ms.
"""),
    ("human", "{problem}"),
])

chain = cot_few | llm | StrOutputParser()
print(chain.invoke({"problem": "Query JOINs users (500K rows) and orders (5M rows) on user_id. Takes 8 seconds. No indexes on join columns."}))
print()

=== 4. Chain-of-Thought (Few-Shot) ===
Step 1 — Identify bottleneck:
- Without any index on the join key, PostgreSQL will pay a heavy cost to match 500K users with 5M orders. Expect nested loop or hash-join with large I/O and sorts.
- Likely plan: full scan of orders (or scans per user) plus many lookups.

Step 2 — Choose the fix:
- Add an index on the join column(s). The join is users.id = orders.user_id, so:
  - Ensure users.id is indexed (PK/unique index exists).
  - Create an index on orders.user_id.
- If you frequently filter on other columns (e.g., order_date, status), you can add a composite index (orders.user_id, ...).

Step 3 — Verify side effects:
- Index adds storage and a small write overhead on inserts/updates to orders.
- After indexing, the planner can use index lookups or a hash join more efficiently, reducing the join time dramatically.

Step 4 — Apply the fix (safe for production):
- Create the index without locking the table:
  CREATE INDEX CONCURRENTLY idx_orders_us

### 5. STRONG SYSTEM PROMPT

In [9]:
print("=== 5. Strong System Prompt ===")
strong_system = ChatPromptTemplate.from_messages([
    ("system", """You are a senior backend security engineer conducting code reviews.

ALWAYS:
- Flag SQL injection, XSS, auth bypass, and missing validation FIRST
- Explain the attack vector, not just "this is insecure"
- Suggest the exact fix in code, not just "use parameterized queries"

NEVER:
- Approve code with SQL string concatenation
- Skip explaining why something is a vulnerability
- Recommend paid security tools when standard library solutions exist

OUTPUT FORMAT:
VERDICT: [APPROVE | APPROVE WITH CHANGES | REJECT]
CRITICAL ISSUES: (security vulnerabilities — fix before merge)
MINOR ISSUES: (non-security improvements)
FIXED CODE: (corrected version)"""),
    ("human", "Review this code snippet:{code}"),
])

chain = strong_system | llm | StrOutputParser()

result = chain.invoke({"code": """
def get_user_by_email(email: str, db):
    query = f"SELECT * FROM users WHERE email = '{email}'"
    return db.execute(query).fetchone()
"""})

print(result)

print()


=== 5. Strong System Prompt ===
VERDICT: APPROVE WITH CHANGES

CRITICAL ISSUES:
- SQL injection vulnerability: The email is interpolated directly into the SQL query using f-string formatting. An attacker could craft an email value that manipulates the query (e.g., "'; DROP TABLE users; --") and execute arbitrary SQL.
- SELECT * usage: Returning all columns can lead to excessive data transfer and potential exposure of sensitive columns. Prefer explicit column selection.

MINOR ISSUES:
- Lack of input validation: Simple type check or email format validation could catch obvious invalid input before hitting the DB.
- Error handling: If the DB call fails, the function may raise an exception that isn’t translated into a meaningful domain error. Consider handling/propagating errors consistently.
- Explicit column list is preferable for maintainability and performance.

FIXED CODE:
def get_user_by_email(email: str, db):
    # Use parameterized queries to prevent SQL injection
    query = "SELE

### 6. TWO-STEP PROMPT CHAIN
*Step 1: Extract requirements from a job description*

*Step 2: Gap analysis against candidate background*

*Run chain sequentially*

In [10]:
print("=== 6. Two-Step Prompt Chain ===")

extract_prompt = ChatPromptTemplate.from_template(
    "Extract the top 5 technical requirements from this job posting as a numbered list.\n"
    "Be specific — include technologies, years of experience, and concepts.\n\n"
    "Job posting: {job_description}"
)

gap_prompt = ChatPromptTemplate.from_template(
    "Given these job requirements:\n{requirements}\n\n"
    "And this candidate background:\n{candidate_background}\n\n"
    "For each requirement, mark it as:\n"
    "✅ MET — candidate clearly has this\n"
    "⚠️ PARTIAL — candidate has related experience but gaps exist\n"
    "❌ MISSING — no evidence of this skill\n\n"
    "End with: OVERALL FIT: X/10 and one sentence recommendation."
)

extract_chain = extract_prompt | llm | StrOutputParser()
gap_chain = gap_prompt | llm | StrOutputParser()

job_description = """
Senior Backend Engineer at Series B startup.
Requirements: 3+ years Python, FastAPI or Django, PostgreSQL with query optimization,
Redis caching, Docker/Kubernetes, REST API design, experience with multi-tenant SaaS.
Bonus: LLM/AI integration experience, Terraform, CI/CD.
"""

candidate_background = """
2 years at Apexon: Django, DRF, PostgreSQL (composite indexes, 40% latency reduction),
Redis caching, Docker, Kubernetes (5 microservices), Terraform/Terragrunt, Azure DevOps CI/CD.
MS thesis: Federated Learning (Python). Currently building FastAPI SaaS API with multi-tenancy,
JWT, RBAC. Learning LangChain and RAG.
"""

requirements = extract_chain.invoke({"job_description": job_description})
print("Extracted Requirements:\n", requirements)
print()

gap_analysis = gap_chain.invoke({
    "requirements": requirements,
    "candidate_background": candidate_background
})
print("Gap Analysis:\n", gap_analysis)
print()


=== 6. Two-Step Prompt Chain ===
Extracted Requirements:
 1) Python backend development (3+ years) with FastAPI or Django
- Technologies: Python, FastAPI or Django
- Concepts: building scalable RESTful services, backend development

2) PostgreSQL with query optimization
- Technologies: PostgreSQL
- Concepts: query optimization, indexing, performance tuning, efficient SQL

3) Redis caching
- Technologies: Redis
- Concepts: caching strategies, data structures (e.g., strings, hashes, sets), TTLs, cache invalidation

4) Docker and Kubernetes
- Technologies: Docker, Kubernetes
- Concepts: containerization, orchestration, deployment scalability, service discovery

5) REST API design for multi-tenant SaaS
- Technologies/Concepts: REST API design, multi-tenant SaaS architecture
- Focus: tenant isolation, authentication/authorization, API versioning, scalable multi-tenant data access

Gap Analysis:
 - 1) Python backend development (3+ years) with FastAPI or Django
  - PARTIAL — has 2 years of D